# Лабораторная 1. Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark

## Описание данных

https://www.kaggle.com/benhamner/sf-bay-area-bike-share

stations.csv схема:

```
id: station ID number
name: name of station
lat: latitude
long: longitude
dock_count: number of total docks at station
city: city (San Francisco, Redwood City, Palo Alto, Mountain View, San Jose)
installation_date: original date that station was installed. If station was moved, it is noted below.
```

trips.csv схема:

```
id: numeric ID of bike trip
duration: time of trip in seconds
start_date: start date of trip with date and time, in PST
start_station_name: station name of start station
start_station_id: numeric reference for start station
end_date: end date of trip with date and time, in PST
end_station_name: station name for end station
end_station_id: numeric reference for end station
bike_id: ID of bike used
subscription_type: Subscriber = annual or 30-day member; Customer = 24-hour or 3-day member
zip_code: Home zip code of subscriber (customers can choose to manually enter zip at kiosk however data is unreliable)
```

In [1]:
from pyspark import SparkContext, SparkConf

In [2]:
conf = SparkConf().setAppName("L1_interactive_bike_analysis").setMaster("yarn")

In [3]:
sc = SparkContext(conf=conf)
sc.setLogLevel('WARN')

In [4]:
tripData = sc.textFile("trips.csv")
# запомним заголовок, чтобы затем его исключить из данных
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile("stations.csv")
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

In [5]:
list(enumerate(tripsHeader.split(",")))

[(0, 'id'), (1, 'duration'), (2, 'start_date'), (3, 'start_station_name'), (4, 'start_station_id'), (5, 'end_date'), (6, 'end_station_name'), (7, 'end_station_id'), (8, 'bike_id'), (9, 'subscription_type'), (10, 'zip_code')]

In [6]:
list(enumerate(stationsHeader.split(",")))

[(0, 'id'), (1, 'name'), (2, 'lat'), (3, 'long'), (4, 'dock_count'), (5, 'city'), (6, 'installation_date')]

In [7]:
trips.take(2)

[['4576', '63', '8/29/2013 14:13', 'South Van Ness at Market', '66', '8/29/2013 14:14', 'South Van Ness at Market', '66', '520', 'Subscriber', '94127'], ['4607', '70', '8/29/2013 14:42', 'San Jose City Hall', '10', '8/29/2013 14:43', 'San Jose City Hall', '10', '661', 'Subscriber', '95138']]

In [8]:
stations.take(2)

[['2', 'San Jose Diridon Caltrain Station', '37.329732', '-121.901782', '27', 'San Jose', '8/6/2013'], ['3', 'San Jose Civic Center', '37.330698', '-121.888979', '15', 'San Jose', '8/5/2013']]

Объявите `stationsIndexed` так, чтобы результатом был список пар ключ-значение с целочисленным ключом  из первой колонки.  Таким образом вы создаёте индекс на основе первой колонки - номера велостоянки

In [9]:
stationsIndexed = stations.keyBy(lambda station: int(station[0]))

In [10]:
stationsIndexed.take(2)

[(2, ['2', 'San Jose Diridon Caltrain Station', '37.329732', '-121.901782', '27', 'San Jose', '8/6/2013']), (3, ['3', 'San Jose Civic Center', '37.330698', '-121.888979', '15', 'San Jose', '8/5/2013'])]

Аналогичное действие проделайте для индексирования коллекции trips по колонкам start_station_id и  end_station_id и сохраните результат в переменные, например tripsByStartTerminals и tripsByEndTerminals.

In [11]:
tripsByStartTerminals = trips.keyBy(lambda trip: int(trip[4]))
tripsByEndTerminals = trips.keyBy(lambda trip: int(trip[7]))

Выполните операцию объединения коллекций по ключу с помощью функции join. Объедините stationsIndexed и tripsByStartTerminals, stationsIndexed и tripsByEndTerminals.

In [12]:
startTrips = stationsIndexed.join(tripsByStartTerminals)
endTrips = stationsIndexed.join(tripsByEndTerminals)

Объявление последовательности трансформаций приводит к созданию ацикличного ориентированного графа. Вывести  полученный граф можно для любого RDD.

In [13]:
print(startTrips.toDebugString().decode("utf-8"))

(8) PythonRDD[18] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[17] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[16] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(8) PairwiseRDD[15] at join at <stdin>:1 []
    |  PythonRDD[14] at join at <stdin>:1 []
    |  trips.csv MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0 []
    |  stations.csv MapPartitionsRDD[7] at textFile at NativeMethodAccessorImpl.java:0 []


In [14]:
print(endTrips.toDebugString().decode("utf-8"))

(8) PythonRDD[18] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[17] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[16] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(8) PairwiseRDD[15] at join at <stdin>:2 []
    |  PythonRDD[14] at join at <stdin>:2 []
    |  trips.csv MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0 []
    |  stations.csv MapPartitionsRDD[7] at textFile at NativeMethodAccessorImpl.java:0 []


Выполните  объявленные графы трансформаций вызовом команды count.

In [15]:
startTrips.count()

669959

In [16]:
endTrips.count()

669959

Если вы знаете распределение ключей заранее, вы можете выбрать оптимальный способ хеширования ключей по разделам `Partition`. Например, если один ключ встречается на порядки чаще, чем другие ключи, то использование `HashPartitioner` будет не лучшим выбором, так как данные связанные с этим ключом будут собираться в одном разделе. Это приведёт к неравномерной нагрузке на вычислительные ресурсы.

Выбрать определённую реализацию класса распределения по разделам можно с помощью функции RDD `partitionBy`. Например, для RDD `stationsIndexed`  выбирается `portable_hash(idx)` с количеством разделов равным количеству разделов trips RDD.

In [17]:
from pyspark.rdd import portable_hash

stationsIndexedPartitioned = stationsIndexed.partitionBy(
    numPartitions=trips.getNumPartitions(),
    partitionFunc=portable_hash
)
stationsIndexedPartitioned.take(2)

[(2, ['2', 'San Jose Diridon Caltrain Station', '37.329732', '-121.901782', '27', 'San Jose', '8/6/2013']), (3, ['3', 'San Jose Civic Center', '37.330698', '-121.888979', '15', 'San Jose', '8/5/2013'])]

Узнать какой класс назначен для текущего RDD можно обращением к полю partitioner.

In [18]:
stationsIndexedPartitioned.partitioner

<pyspark.rdd.Partitioner object>

## Создание модели данных

Для более эффективной  обработки и получения дополнительных возможностей мы можем объявить классы сущностей предметной области и преобразовать исходные строковые данные в объявленное представление.

In [19]:
from typing import NamedTuple
from datetime import datetime

In [20]:
def initStation(stations):
    class Station(NamedTuple):
        station_id: int
        name: str
        lat: float
        long: float
        dockcount: int
        landmark: str
        installation: str
    
    for station in stations:
        yield Station(
            station_id = int(station[0]),
            name = station[1],
            lat = float(station[2]),
            long = float(station[3]),
            dockcount = int(station[4]),
            landmark = station[5],
            installation = datetime.strptime(station[6], '%m/%d/%Y')
        )

In [21]:
stationsInternal = stations.mapPartitions(initStation)
stationsInternal.first()

Station(station_id=2, name='San Jose Diridon Caltrain Station', lat=37.329732, long=-121.901782, dockcount=27, landmark='San Jose', installation=datetime.datetime(2013, 8, 6, 0, 0))

In [22]:
def initTrip(trips):
    class Trip(NamedTuple):
        trip_id: int
        duration: int
        start_date: datetime
        start_station_name: str
        start_station_id: int
        end_date: datetime
        end_station_name: str
        end_station_id: int
        bike_id: int
        subscription_type: str
        zip_code: str

    for trip in trips:
        try:
            yield Trip(
                trip_id=int(trip[0]),
                duration=int(trip[1]),
                start_date=datetime.strptime(trip[2], '%m/%d/%Y %H:%M'),
                start_station_name=trip[3],
                start_station_id=int(trip[4]),
                end_date=datetime.strptime(trip[5], '%m/%d/%Y %H:%M'),
                end_station_name=trip[6],
                end_station_id=int(trip[7]),
                bike_id=int(trip[8]),
                subscription_type=trip[9],
                zip_code=trip[10]
            )
        except Exception:
            pass


In [23]:
tripsInternal = trips.mapPartitions(initTrip)
tripsInternal.take(10)

[Trip(trip_id=4576, duration=63, start_date=datetime.datetime(2013, 8, 29, 14, 13), start_station_name='South Van Ness at Market', start_station_id=66, end_date=datetime.datetime(2013, 8, 29, 14, 14), end_station_name='South Van Ness at Market', end_station_id=66, bike_id=520, subscription_type='Subscriber', zip_code='94127'), Trip(trip_id=4607, duration=70, start_date=datetime.datetime(2013, 8, 29, 14, 42), start_station_name='San Jose City Hall', start_station_id=10, end_date=datetime.datetime(2013, 8, 29, 14, 43), end_station_name='San Jose City Hall', end_station_id=10, bike_id=661, subscription_type='Subscriber', zip_code='95138'), Trip(trip_id=4130, duration=71, start_date=datetime.datetime(2013, 8, 29, 10, 16), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 10, 17), end_station_name='Mountain View City Hall', end_station_id=27, bike_id=48, subscription_type='Subscriber', zip_code='97214')]

Для каждой стартовой станции найдем среднее время поездки. Будем использовать метод groupByKey.

Для этого потребуется преобразовать trips RDD в RDD коллекцию пар ключ-значение аналогично тому, как мы совершали это ранее методом keyBy.

In [24]:
tripsByStartStation = tripsInternal.keyBy(lambda trip: trip.start_station_name)

Рассчитаем среднее время поездки для каждого стартового парковочного места

In [25]:
import numpy as np

avgDurationByStartStation = tripsByStartStation\
 .mapValues(lambda trip: trip.duration)\
 .groupByKey()\
 .mapValues(lambda trip_durations: np.mean(list(trip_durations)))

Выведем первые 10 результатов

In [26]:
%%time

avgDurationByStartStation.top(10, key=lambda x: x[1])

CPU times: user 52 ms, sys: 9 ms, total: 61 ms
Wall time: 3.42 s


[('San Jose Government Center', 7146.5), ('Santa Clara County Civic Center', 5934.2), ('Broadway St at Battery St', 1827.4), ('Park at Olive', 1471.6), ('Mezes Park', 1440.3), ('Arena Green / SAP Center', 1418.5), ('San Jose City Hall', 1132.0), ('Golden Gate at Polk', 1039.7), ('Paseo de San Antonio', 1019.8), ('SJSU 4th at San Carlos', 995.4)]

Выполнение операции groupByKey приводит к интенсивным передачам данных. Если группировка делается для последующей редукции элементов лучше использовать трансформацию reduceByKey или aggregateByKey. Их выполнение приведёт сначала к локальной редукции над разделом Partition, а затем будет произведено окончательное суммирование над полученными частичными суммами.

*Примечание.* Выполнение reduceByKey логически сходно с выполнением Combine и Reduce фазы MapReduce  работы.

Функция aggregateByKey является аналогом reduceByKey с возможностью указывать начальный элемент.

Рассчитаем среднее значение с помощью aggregateByKey. Одновременно будут вычисляться два значения для каждого стартового терминала: сумма времён и количество поездок.

In [27]:
? tripsByStartStation.aggregateByKey

Type:        method
String form: <bound method RDD.aggregateByKey of PythonRDD[31] at RDD at PythonRDD.scala:53>
File:        /opt/mapr/spark/spark/python/pyspark/rdd.py


In [28]:
def seqFunc(acc, duration):
    duration_sum, count = acc
    return (duration_sum + duration, count + 1)

def combFunc(acc1, acc2):
    duration_sum1, count1 = acc1
    duration_sum2, count2 = acc2
    return (duration_sum1+duration_sum2, count1+count2)

def meanFunc(acc):
    duration_sum, count = acc
    return duration_sum/count

avgDurationByStartStation2 = tripsByStartStation\
  .mapValues(lambda trip: trip.duration)\
  .aggregateByKey(
    zeroValue=(0,0),
    seqFunc=seqFunc,
    combFunc=combFunc)\
  .mapValues(meanFunc)

В `zeroValue` передаётся начальное значение. В нашем случае это пара нулей. Первая функция `seqFunc` предназначена для прохода по коллекции партиции. На этом проходе значение элементов помещаются средой в переменную duration, а переменная «аккумулятора» acc накапливает значения. Вторая функция `combFunc` предназначена для этапа редукции частично посчитанных локальных результатов.

Сравните результаты `avgDurationByStartStation` и `avgDurationByStartStation2` и их время выполнения.

In [29]:
%%time

avgDurationByStartStation2.top(10, key=lambda x: x[1])

CPU times: user 38 ms, sys: 6 ms, total: 44 ms
Wall time: 1.67 s


[('San Jose Government Center', 7146.5), ('Santa Clara County Civic Center', 5934.2), ('Broadway St at Battery St', 1827.4), ('Park at Olive', 1471.6), ('Mezes Park', 1440.3), ('Arena Green / SAP Center', 1418.5), ('San Jose City Hall', 1132.0), ('Golden Gate at Polk', 1039.7), ('Paseo de San Antonio', 1019.8), ('SJSU 4th at San Carlos', 995.4)]

Теперь найдём первую поездку для каждой велостоянки. Для решения опять потребуется группировка. Ещё одним недостатком `groupByKey` данных является то, что для группировки данные должны поместиться в оперативной памяти. Это может привести к ошибке `OutOfMemoryException` для больших объёмов данных.

Найдем самую раннюю поездку для каждой станции. Сгруппируем поездки по станциям, возьмём первую поездку из отсортированного списка:

In [30]:
def earliestTrip(trips):
    trips = list(trips)
    if not trips:
        return None
    min_trip = trips[0]
    for trip in trips[1:]:
        if trip.start_date < min_trip.start_date:
            min_trip = trip
    return min_trip

firstGrouped = tripsByStartStation\
  .groupByKey()\
  .mapValues(earliestTrip)


In [31]:
%%time

firstGrouped.take(5)

CPU times: user 41 ms, sys: 5 ms, total: 46 ms
Wall time: 2.71 s


[('South Van Ness at Market', Trip(trip_id=4576, duration=63, start_date=datetime.datetime(2013, 8, 29, 14, 13), start_station_name='South Van Ness at Market', start_station_id=66, end_date=datetime.datetime(2013, 8, 29, 14, 14), end_station_name='South Van Ness at Market', end_station_id=66, bike_id=520, subscription_type='Subscriber', zip_code='94127')), ('San Jose City Hall', Trip(trip_id=4607, duration=70, start_date=datetime.datetime(2013, 8, 29, 14, 42), start_station_name='San Jose City Hall', start_station_id=10, end_date=datetime.datetime(2013, 8, 29, 14, 43), end_station_name='San Jose City Hall', end_station_id=10, bike_id=661, subscription_type='Subscriber', zip_code='95138'))]

Лучшим вариантом с точки зрения эффективности будет использование трансформации `reduceByKey`

In [32]:
firstGrouped = tripsByStartStation\
  .reduceByKey(lambda tripA, tripB: tripA if tripA.start_date < tripB.start_date else tripB)

In [33]:
%%time

firstGrouped.take(5)

CPU times: user 29 ms, sys: 4 ms, total: 33 ms
Wall time: 1.11 s


[('South Van Ness at Market', Trip(trip_id=4576, duration=63, start_date=datetime.datetime(2013, 8, 29, 14, 13), start_station_name='South Van Ness at Market', start_station_id=66, end_date=datetime.datetime(2013, 8, 29, 14, 14), end_station_name='South Van Ness at Market', end_station_id=66, bike_id=520, subscription_type='Subscriber', zip_code='94127')), ('San Jose City Hall', Trip(trip_id=4607, duration=70, start_date=datetime.datetime(2013, 8, 29, 14, 42), start_station_name='San Jose City Hall', start_station_id=10, end_date=datetime.datetime(2013, 8, 29, 14, 43), end_station_name='San Jose City Hall', end_station_id=10, bike_id=661, subscription_type='Subscriber', zip_code='95138'))]

## Решение задач `L1_Apache_Spark_Tasks.md`

Ниже приведено решение 5 обязательных задач для данных `trips.csv` и `stations.csv` через RDD API.

In [34]:
from math import atan2, cos, radians, sin, sqrt

THREE_HOURS_SECONDS = 3 * 60 * 60

def format_duration(total_seconds):
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0088
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return radius * c


In [35]:
bikeRuntime = tripsInternal\
    .map(lambda trip: (trip.bike_id, int(trip.duration)))\
    .reduceByKey(lambda left, right: left + right)

maxBikeId, maxBikeTotalDuration = bikeRuntime.takeOrdered(1, key=lambda item: -item[1])[0]

stationPairs = stationsInternal.cartesian(stationsInternal)\
    .filter(lambda pair: pair[0].station_id < pair[1].station_id)

maxDistance = stationPairs\
    .map(lambda pair: (
        haversine_km(pair[0].lat, pair[0].long, pair[1].lat, pair[1].long),
        pair[0],
        pair[1]
    ))\
    .takeOrdered(1, key=lambda item: -item[0])[0]

maxBikeTrips = tripsInternal\
    .filter(lambda trip: trip.bike_id == maxBikeId)\
    .sortBy(lambda trip: (trip.start_date, trip.trip_id))\
    .collect()

bikePath = "path not found" if not maxBikeTrips else " -> ".join(
    [maxBikeTrips[0].start_station_name] + [trip.end_station_name for trip in maxBikeTrips]
)

bikesCount = tripsInternal.map(lambda trip: trip.bike_id).distinct().count()

heavyUsers = tripsInternal\
    .filter(lambda trip: trip.zip_code is not None and trip.zip_code.strip() != "" and trip.zip_code.strip().lower() != "nil")\
    .map(lambda trip: (trip.zip_code.strip(), int(trip.duration)))\
    .reduceByKey(lambda left, right: left + right)\
    .filter(lambda item: item[1] > THREE_HOURS_SECONDS)\
    .sortBy(lambda item: item[1], ascending=False)


In [36]:
print("1. Велосипед с максимальным временем пробега")
print(f"bikeId = {maxBikeId}")
print(f"Суммарное время = {format_duration(maxBikeTotalDuration)} ({maxBikeTotalDuration} сек.)")

print("\n2. Наибольшее геодезическое расстояние между станциями")
print(f"Станция 1: {maxDistance[1].name} (#{maxDistance[1].station_id})")
print(f"Станция 2: {maxDistance[2].name} (#{maxDistance[2].station_id})")
print(f"Расстояние = {maxDistance[0]:.3f} км")

print("\n3. Путь велосипеда с максимальным временем пробега через станции")
print(f"bikeId = {maxBikeId}")
print(f"Количество поездок этого велосипеда = {len(maxBikeTrips)}")
print(bikePath[:2000] + (" ..." if len(bikePath) > 2000 else ""))

print("\n4. Количество велосипедов в системе")
print(f"Количество велосипедов = {bikesCount}")

print("\n5. Пользователи, потратившие на поездки более 3 часов")
print("Примечание: в качестве идентификатора пользователя используется zipCode, потому что отдельного userId в наборе данных нет.")
for zip_code, total_duration in heavyUsers.take(20):
    print(f"zipCode = {zip_code}, суммарное время = {format_duration(total_duration)} ({total_duration} сек.)")


1. Велосипед с максимальным временем пробега
bikeId = 535
Суммарное время = 5169:54:53 (18611693 сек.)

2. Наибольшее геодезическое расстояние между станциями
Станция 1: SJSU - San Salvador at 9th (#16)
Станция 2: Embarcadero at Sansome (#60)
Расстояние = 69.921 км

3. Путь велосипеда с максимальным временем пробега через станции
bikeId = 535
Количество поездок этого велосипеда = 1328
Post at Kearney -> San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome -> 2nd at South Park -> Davis at Jackson -> Civic Center BART (7th at Market) -> Post at Kearney -> Embarcadero at Sansome -> Washington at Kearney -> Market at Sansome -> Market at Sansome -> 2nd at Folsom -> 2nd at Townsend -> 2nd at Townsend -> Embarcadero at Sansome -> Clay at Battery -> Harry Bridges Plaza (Ferry Building) -> Clay at Battery -> San Francisco Caltrain (Townsend at 4th) -> Steuart at Market -> 2nd at Townsend -> Harry Bridges Plaza (Ferry Building) -> Townsend at 

In [37]:
sc.stop()